# PydanticAI together with Pydantic Models

In [3]:
from model import nvidia_model
from pydantic_ai import Agent

employee_simulator = Agent(
    model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions="Du är expert på att simulera anställda i Sverige. Simulera i detta formatet likt exemplet"
    """
        name: "Bertil",
        age: 40,
        salary: 35000,
        role: "IT-supporter",
        ort: "Göteborg"
    """
    )

result = await employee_simulator.run("Ge mig en data engineer som jobbar i stockholm ")
print(result.output)

                 pydantic-ai v2.63.0 • Python 3.13.7

      / \        agent: employee_simulator • model: openai:nvidia/nemotron-3.5-super-vl-120b-a12b
     /   \         tools: 0 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


        name: "Erik Andersson",
        age: 34,
        salary: 58000,
        role: "Data Engineer",
        ort: "Stockholm"


## Force a structured output using Pydantic

In [ ]:
from pydantic import BaseModel, Field


class EmployeeModel(BaseModel):
    name: str = Field(
        description="för och efternamn, inte jättelångt, svenska eller internationella namn"
    )
    age: int = Field(description="ålder från 18 till 67 år", ge=18, le=67)
    salary: int = Field(
        description="lön mellan 30k-60k, ju mer senior position, desto högre lön",
        ge=30_000,
        lt=60_000,
    )
    role: str = Field(description="yrke/roll")
    description: str = Field(description="kort beskrivning, 2 meningar på rollen")


employee1 = EmployeeModel(
    name="Bertil",
    age=40,
    salary=30000,
    role="IT supporter",
    description="Hjälper folk med IT",
)

employee1

EmployeeModel(name='Bertil', age=40, salary=30000, role='IT supporter', description='Hjälper folk med IT')

In [5]:
employee_simulator = Agent(
    model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions="Du är expert på att simulera anställda i Sverige",
)

result = await employee_simulator.run("Ge mig en data engineer i stockholm")

result.output

'Här är en simulerad personlighet: en data engineer som arbetar i Stockholm. Allt är fiktivt, men byggt på verkliga mönster från den svenska tech- och konsultbranschen.\n\n---\n\n### Personlig presentation\n\n**Namn:** Amanda Karlsson  \n**Ålder:** 29  \n**Roll:** Data Engineer  \n**Plats:** Stockholm (Södermalm, hybridarbete 2–3 dagar i veckan)  \n**Arbetsplats:** En snabbväxande climate-tech scale-up med ca 120 anställda, kontoret ligger nära Medborgarplatsen. Kontorsspråket är engelska, men det svenska samhället och avtalen styr vardagen.\n\n---\n\n### Bakgrund och utbildning\n\n- **Utbildning:** Civilingenjör i datateknik, KTH Royal Institute of Technology (examen 2019). Specialiserad på storskaliga datapipelines och molnarkitektur.\n- **Erfarenhet:** 5 år som data engineer. Började på en stor konsultfirma i Kista, bytte till ett produktbolag för två år sedan för att komma närmare affären och dataprodukten.\n- **Språk:** Flytande svenska och engelska. Talar obehindrat på möten och 

In [6]:
result = await employee_simulator.run(
    "Ge mig en data engineer i stockholm", output_type=EmployeeModel
)
result

AgentRunResult(output=EmployeeModel(name='Erik Andersson', age=32, salary=48000, role='Data Engineer', description='Erik arbetar med att designa och underhålla datapipelines och molnbaserade datalager för ett medelstort techbolag i Stockholm. Han brinner för att omvandla rådata till tillgängliga och pålitliga dataprodukter som stöder affärsbeslut.'))

In [7]:
employee2 = result.output
employee2

EmployeeModel(name='Erik Andersson', age=32, salary=48000, role='Data Engineer', description='Erik arbetar med att designa och underhålla datapipelines och molnbaserade datalager för ett medelstort techbolag i Stockholm. Han brinner för att omvandla rådata till tillgängliga och pålitliga dataprodukter som stöder affärsbeslut.')

In [8]:
type(employee2)

__main__.EmployeeModel

In [9]:
employee2.name + " är cool"

'Erik Andersson är cool'

In [10]:
employee2.salary

48000

In [11]:
employee2.__dict__.keys()

dict_keys(['name', 'age', 'salary', 'role', 'description'])

In [12]:
employee2.__dict__.values()

dict_values(['Erik Andersson', 32, 48000, 'Data Engineer', 'Erik arbetar med att designa och underhålla datapipelines och molnbaserade datalager för ett medelstort techbolag i Stockholm. Han brinner för att omvandla rådata till tillgängliga och pålitliga dataprodukter som stöder affärsbeslut.'])

In [13]:
for key, value in employee2.__dict__.items():
    print(f"{key}: {value}")

name: Erik Andersson
age: 32
salary: 48000
role: Data Engineer
description: Erik arbetar med att designa och underhålla datapipelines och molnbaserade datalager för ett medelstort techbolag i Stockholm. Han brinner för att omvandla rådata till tillgängliga och pålitliga dataprodukter som stöder affärsbeslut.


## Simulate many employees

In [14]:
result = await employee_simulator.run("Ge mig totalt 10 anställda inom IT/HR/marketing",
                                      output_type=list[EmployeeModel])
result

AgentRunResult(output=[EmployeeModel(name='Erik Johansson', age=28, salary=38000, role='Systemutvecklare', description='Utvecklar och underhåller webbapplikationer för företagets kunder. Arbetar främst med JavaScript och molnbaserade lösningar.'), EmployeeModel(name='Sara Nilsson', age=34, salary=45000, role='IT-supporttekniker', description='Hjälper kollegor med datorproblem och installationer av mjukvara. Ansvarar även för företagets IT-inventarier.'), EmployeeModel(name='David Karlsson', age=42, salary=56000, role='Molnarkitekt', description='Designar och implementerar skalbara molnlösningar på Azure och AWS. Säkerställer säkerhet och driftsäkerhet i företagets IT-miljö.'), EmployeeModel(name='Linda Berg', age=39, salary=52000, role='IT-säkerhetsspecialist', description='Overvakar nätverkstrafik och utreder potentiella säkerhetsincidenter. Genomför regelbundna säkerhetsutbildningar för personalen.'), EmployeeModel(name='Maria Andersson', age=31, salary=36000, role='HR-assistent', de